#### Note: This colab only contains short discussion/highlights of task done. Please visit the google document for my detailed writeup:
Google document link:

### TASK 1: Understanding MaxText Data format

To complete task 1 I explored files inside folder `maxtext/input_pipeline` , especially `input_pipeline_inferace.py`.

Here are the dataset formats supported by MaxText:

- `synthetic dataset`: fake dataset (token ids) batches are created in memory, rather than loading any dataset. Used for bechmarkiing hardware throughput purposes etc.

- `grain`: you can load the dataset using following file types:    
  - `arrayrecord`: ArrayRecord file format supports parallel read, write, and random access by record index.

  - `parquet`: Apache parquet format - open-source columnar storage format.


  - `tfrecord`: TensorFlow's standard format for storing a sequence of binary records.




### TASK 2:



To complete task 2 let's first look at the resources available.

In [ ]:

import os, psutil, platform

print("=== Hardware Info ===")
print(f"CPU cores: {os.cpu_count()}")
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
print(f"Platform: {platform.platform()}")


import os
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
print(f"\nJAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

Let's answer the most important question: how much memory is occupied when we train the model and what actually require memory ?

Training requires forward and backward pass:
this is what consume memory:
- Model parameters
- gradients
- optimizer states
- also one thing to note: activations stored from forward pass for backward pass also requires memory.

What also matters is if your model parameter's precision format is in FP32 (32 bit), FP16 (16 bits) or is it quantized (8 bit, 4bit)

For optimizer states, usually it is in fp32

In [ ]:
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext


Cloning into 'maxtext'...
Updating files:  21% (256/1211)
Updating files:  22% (267/1211)
Updating files:  23% (279/1211)
Updating files:  24% (291/1211)
Updating files:  25% (303/1211)
Updating files:  26% (315/1211)
Updating files:  27% (327/1211)
Updating files:  28% (340/1211)
Updating files:  29% (352/1211)
Updating files:  30% (364/1211)
Updating files:  31% (376/1211)
Updating files:  32% (388/1211)
Updating files:  33% (400/1211)
Updating files:  33% (404/1211)
Updating files:  34% (412/1211)
Updating files:  35% (424/1211)
Updating files:  36% (436/1211)
Updating files:  37% (449/1211)
Updating files:  38% (461/1211)
Updating files:  39% (473/1211)
Updating files:  40% (485/1211)
Updating files:  41% (497/1211)
Updating files:  42% (509/1211)
Updating files:  43% (521/1211)
Updating files:  44% (533/1211)
Updating files:  45% (545/1211)
Updating files:  46% (558/1211)
Updating files:  47% (570/1211)
Updating files:  48% (582/1211)
Updating files:  49% (594/1211)
Updating files

In [4]:
!uv venv --python 3.12 --seed exp
!source exp/bin/activate

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: exp
 + pip==26.1.2
Activate with: source exp/bin/activate


In [ ]:
!pip install -r src/dependencies/requirements/requirements.txt
!pip install -e .

In [3]:
# ## Using attention=dot_product (by default it is autoselected which might choose flash attention). flash attention kernels probably
# ## might be written in triton and Triton native GPU kernel execution failed on Turing architecture (T4) and requires ampere architecture
# ## hence using attention=dot_product
# # Then run training


## decoder layers = 28, embd= 1024, ml_dim = 3072

!python3 -m maxtext.trainers.pre_train.train \
  /content/maxtext/src/maxtext/configs/base.yml \
  model_name=qwen3-0.6b \
  dataset_type=synthetic \
  override_model_config=True \
  hardware=cpu \
  base_num_decoder_layers=18 \
  reuse_example_batch=1 \
  per_device_batch_size=1 \
  max_target_length=512 \
  base_output_directory=/tmp/maxtext_runs \
  run_name=qwen3_0.6_cpu \
  enable_checkpointing=False \
  skip_jax_distributed_system=True \
  attention=dot_product \
  steps=50 \
  mu_dtype=bfloat16 \
  2>&1 | tee /tmp/training_log_cpu_0.6.txt

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-06-19 10:15:36.943009: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-19 10:15:38.387532: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-19 10:15:42.022253: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn'